# Series compartments and the effect of latency

Adapted from the [Monash EMU summer textbook](https://github.com/monash-emu/summer-textbook)
notebook `textbook/05-latency-and-series-comps.ipynb` at commit
`fd97783474789e50ace5ea420aec20147f9bbd76`.

Source licence: BSD-2-Clause, Copyright (c) 2022, monash-emu. Prose is carried
and adapted; code is written in summer4 idiom.

This notebook looks at the effect on a simple SIR / SEIR epidemic of including
one or two sequential compartments for the latent period after infection but
before infectiousness. It then considers the more general effect of chaining
many compartments in series.

## Terminology

The **latent** (pre-infectious) period is the time from infection to the onset
of infectiousness. The **incubation** period is the time from infection to
symptom onset. In a plain SIR model there is no delay between infection and
infectiousness, so the latent period is zero. Symptoms are not represented
explicitly, so there is no incubation period either — symptom status only
matters when symptoms change epidemiology (for example isolation).

The **serial interval** and **generation time** depend on these periods and on
the infectious period. The serial interval is the time from symptom onset in
one case to symptom onset in a person they infect; the generation time is the
analogous interval measured from infection to infection.

![](figures/05/incubation_terminology.svg)

## Effect on epidemic dynamics

An intervening latency compartment delays the initial take-off of an epidemic,
with only marginal effect on final size. Extra latency compartments in series
have a milder effect on dynamics but can better represent a true delay from
infection to infectiousness. Whether to include them depends on the question.


In [ ]:
from typing import Any

import numpy as np
import pandas as pd
import plotly.io as pio
from scipy.stats import erlang

from summer4 import Compartments, Param, Property, PropertyMap, SavePlan, SaveRequest, TransitionFlow, FlowModel
from summer4.epi import ForceOfInfection, MixingMatrix

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

AXIS_I = {"index": "time", "value": "infectious prevalence"}
AXIS_R = {"index": "time", "value": "proportion recovered"}
AXIS = {"index": "time", "value": "proportion"}


def infectious_series(res: Any, state: Property) -> pd.Series:
    frame = res["comp"].select(state["infectious"]).to_pandas()
    return frame.iloc[:, 0]


def recovered_series(res: Any, state: Property) -> pd.Series:
    frame = res["comp"].select(state["recovered"]).to_pandas()
    return frame.iloc[:, 0]


In [ ]:
def build_latency_model(
    disease_states: tuple[str, ...],
    *,
    infection_dest: str,
    progression_edges: list[tuple[str, str, object]] | None = None,
) -> tuple[Any, PropertyMap, Property]:
    """SIR/SEIR-style model with frequency-dependent infection into ``infection_dest``."""
    state = Property("state", disease_states)
    pop = Property("pop", ("all",))
    pmap = PropertyMap.from_property(state).stratify(pop)
    model = FlowModel(pmap)
    
    mixing = MixingMatrix(pop, np.array([[1.0]]), check_reciprocal=False)
    model.add_flow(
    TransitionFlow(
        "infection",
        state["susceptible"],
        state[infection_dest],
        ForceOfInfection(
    "infection",
    infectious=state["infectious"],
    group_by=pop,
    kind="frequency",
    contact_rate=Param("contact_rate"),
    mixing=mixing,
),
    )
)
    if progression_edges:
        for i, (src, dest, rate) in enumerate(progression_edges):
            model.add_flow(TransitionFlow(f"progression_{i}", state[src], state[dest], rate))
    model.add_flow(TransitionFlow(
        "recovery",
        state["infectious"],
        state["recovered"],
        Param("recovery_rate"),
    ))
    return model.compile(), pmap, state


model_config = {
    "population": 1.0,
    "seed": 0.001,
    "end_time": 50.0,
}
parameters = {
    "contact_rate": 1.0,
    "recovery_rate": 0.333,
    "progression": 1.0,
}
times = np.linspace(0.0, model_config["end_time"], 501)
plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=times)


def seed_y0(pmap: PropertyMap, state: Property) -> np.ndarray:
    y0 = np.zeros(pmap.size)
    y0[pmap.select(state["susceptible"])] = model_config["population"] - model_config["seed"]
    y0[pmap.select(state["infectious"])] = model_config["seed"]
    return y0


def run_epi(cm: Any, pmap: PropertyMap, state: Property, params: dict[str, float]) -> Any:
    return cm.run(
        params,
        seed_y0(pmap, state),
        t0=0.0,
        t1=model_config["end_time"],
        dt=0.1,
        save=plan,
        solver="dopri5",
    )


### SIR baseline (no latency)

Infection moves people straight from susceptible to infectious.


In [ ]:
sir_cm, sir_pmap, sir_state = build_latency_model(
    ("susceptible", "infectious", "recovered"),
    infection_dest="infectious",
)
sir_res = run_epi(sir_cm, sir_pmap, sir_state, parameters)
sir_i = infectious_series(sir_res, sir_state)
sir_r = recovered_series(sir_res, sir_state)


### SEIR — one exposed compartment

![](figures/05/seir_structure.svg)


In [ ]:
seir_cm, seir_pmap, seir_state = build_latency_model(
    ("susceptible", "exposed", "infectious", "recovered"),
    infection_dest="exposed",
    progression_edges=[("exposed", "infectious", Param("progression"))],
)
seir_res = run_epi(seir_cm, seir_pmap, seir_state, parameters)
seir_i = infectious_series(seir_res, seir_state)
seir_r = recovered_series(seir_res, seir_state)


### SEEIR — two exposed compartments in series

Rates through each latent compartment are scaled by the number of stages so the
mean total latent sojourn stays $1/\mathrm{progression}$.

![](figures/05/seeir_structure.svg)


In [ ]:
n_latent = 2
seeir_cm, seeir_pmap, seeir_state = build_latency_model(
    ("susceptible", "exposed_0", "exposed_1", "infectious", "recovered"),
    infection_dest="exposed_0",
    progression_edges=[
        ("exposed_0", "exposed_1", Param("progression") * n_latent),
        ("exposed_1", "infectious", Param("progression") * n_latent),
    ],
)
seeir_res = run_epi(seeir_cm, seeir_pmap, seeir_state, parameters)
seeir_i = infectious_series(seeir_res, seeir_state)
seeir_r = recovered_series(seeir_res, seeir_state)


### Peak timing differs when latency is included


In [ ]:
infectious_compare = pd.DataFrame({"sir": sir_i, "seir": seir_i, "seeir": seeir_i})
infectious_compare.plot(labels=AXIS_I, title="Infectious prevalence with and without latency")

peak_times = {name: float(series.idxmax()) for name, series in infectious_compare.items()}
assert peak_times["seir"] > peak_times["sir"], "SEIR peak should arrive later than SIR"
assert peak_times["seeir"] > peak_times["sir"], "SEEIR peak should arrive later than SIR"
assert float(sir_i.max()) > model_config["seed"]


### Final size is very similar


In [ ]:
recovered_compare = pd.DataFrame({"sir": sir_r, "seir": seir_r, "seeir": seeir_r})
recovered_compare.plot(labels=AXIS_R, title="Recovered proportion (final size proxy)")

finals = recovered_compare.iloc[-1]
assert float(finals.max() - finals.min()) < 0.05, "latency should barely change final size"


## Multi-compartment chains without transmission

Next, drop transmission and put everyone in the first of $n$ compartments in
series. To keep the mean time to the last compartment equal across
configurations, multiply each stage rate by $(n-1)$. We use a mean sojourn of
30 time units.


In [ ]:
def build_series_comps_model(n_comps: int) -> tuple[Any, PropertyMap, Property]:
    names = tuple(f"comp_{i}" for i in range(n_comps))
    state = Property("state", names)
    pmap = PropertyMap.from_property(state)
    model = FlowModel(pmap)
    progression_rate = (1.0 / Param("sojourn_time")) * (n_comps - 1)
    for i in range(n_comps - 1):
        model.add_flow(
            TransitionFlow(
                f"progression_{i}",
                state[f"comp_{i}"],
                state[f"comp_{i + 1}"],
                progression_rate,
            )
        )
    return model.compile(), pmap, state


series_params = {"sojourn_time": 30.0}
# Representative chain lengths (source goes denser up to ~100; the shape is the same).
n_comp_requests = [2, 3, 4, 6, 8, 10, 20, 40, 60]
series_times = np.linspace(0.0, 60.0, 121)
series_plan = SavePlan(requests={"comp": SaveRequest(Compartments())}, ts=series_times)

outputs: dict[int, pd.Series] = {}
for n_comps in n_comp_requests:
    cm, pmap, state = build_series_comps_model(n_comps)
    y0 = np.zeros(pmap.size)
    y0[pmap.select(state["comp_0"])] = 1.0
    res = cm.run(
        series_params,
        y0,
        t0=0.0,
        t1=60.0,
        dt=0.5,
        save=series_plan,
        solver="dopri5",
    )
    last = state[f"comp_{n_comps - 1}"]
    outputs[n_comps] = res["comp"].select(last).to_pandas().iloc[:, 0]

outputs_df = pd.DataFrame(outputs)
outputs_df.plot(labels=AXIS, title="Size of the last compartment in the chain")
assert float(outputs_df[2].iloc[-1]) > 0.5
assert float(outputs_df[60].iloc[10]) < float(outputs_df[2].iloc[10]), (
    "more stages should delay early arrivals into the last compartment"
)


## Equivalent mathematical distribution

Chaining compartments like this implements an **Erlang** delay (a gamma
distribution with integer shape). Shape 1 is a single exponential transition;
shape 2 is one intervening compartment; shape 3 is two intervening compartments;
and so on.

The difference of the last-compartment trajectory approximates the arrival-time
density; compare that to the Erlang PDF with the same mean sojourn.


In [ ]:
diff_df = outputs_df.diff().fillna(0.0)
diff_df.plot(labels=AXIS, title="Modelled transition time (finite difference)")

erlang_pdfs = {}
for n_comps in n_comp_requests:
    shape = n_comps - 1
    erlang_pdfs[shape] = erlang.pdf(
        series_times,
        shape,
        scale=series_params["sojourn_time"] / shape,
    )

erlang_df = pd.DataFrame(erlang_pdfs, index=series_times)
erlang_df.plot(labels=AXIS, title="Equivalent Erlang PDFs")

# Peak of the arrival density should move later as shape increases.
peak_2 = float(diff_df[2].idxmax())
peak_20 = float(diff_df[20].idxmax())
assert peak_20 > peak_2
